# Silver - Squad 3

## Tabelas de saída
- physical_lojas
- physical_produtos_pereciveis

## Processamento
- Origem principal: camada Bronze
- Aplicação das regras de qualidade e negócio atribuídas às tabelas
- Inclusão de silver_processed_at
- Particionamento por ano e mês do processamento Silver
- Persistência em Delta
- Atualizações incrementais através de MERGE

## Fontes auxiliares
Quando uma regra depende de outra tabela:
1. Priorizar a camada Silver quando disponível.
2. Utilizar temporariamente a Raw quando a Silver dependente ainda não estiver disponível.

## Observação
As regras que explicitamente dependem da camada Gold serão documentadas
neste notebook e implementadas na etapa Gold.

In [0]:
%python

from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    current_timestamp,
    year,
    month,
    trim,
    upper,
    regexp_replace,
    length,
    countDistinct,
    avg,
    sum as spark_sum,
    count
)

from delta.tables import DeltaTable

In [0]:
# Credenciais Azure
# Não versionar credenciais reais no Git

client_id = "<AZURE_CLIENT_ID>"
tenant_id = "<AZURE_TENANT_ID>"
client_secret = "<AZURE_CLIENT_SECRET>"

storage_account_name = "internshipdatalake"

In [0]:
%python

config = {
    "fs.azure.account.auth.type": "OAuth",

    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    "fs.azure.account.oauth2.client.id":
        client_id,

    "fs.azure.account.oauth2.client.secret":
        client_secret,

    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("Configuração ADLS2 carregada.")

In [0]:
%python

base_squad3 = (
    "abfss://squad3@internshipdatalake.dfs.core.windows.net/"
)

# ============================================================
# BRONZE - tabelas principais
# ============================================================

bronze_lojas = (
    base_squad3 +
    "bronze/physical_lojas"
)

bronze_produtos = (
    base_squad3 +
    "bronze/physical_produtos_pereciveis"
)


# ============================================================
# SILVER - tabelas de saída
# ============================================================

silver_lojas = (
    base_squad3 +
    "silver/physical_lojas"
)

silver_produtos = (
    base_squad3 +
    "silver/physical_produtos_pereciveis"
)

In [0]:
%python

def salvar_silver_merge(
    df,
    caminho_silver,
    chave
):

    # Verifica se já existe uma tabela Delta Silver
    if DeltaTable.isDeltaTable(
        spark,
        caminho_silver
    ):

        delta_target = DeltaTable.forPath(
            spark,
            caminho_silver
        )

        (
            delta_target.alias("target")
                .merge(
                    df.alias("source"),
                    f"target.{chave} = source.{chave}"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
        )

        print(
            f"MERGE realizado: {caminho_silver}"
        )

    else:

        (
            df.write
                .format("delta")
                .mode("overwrite")
                .options(**config)
                .partitionBy(
                    "ano",
                    "mes"
                )
                .save(caminho_silver)
        )

        print(
            f"Tabela Silver criada: {caminho_silver}"
        )

In [0]:
%python

df_lojas_bronze = (
    spark.read
        .format("delta")
        .options(**config)
        .load(bronze_lojas)
)

print(
    "Bronze physical_lojas:",
    df_lojas_bronze.count()
)

df_lojas_bronze.printSchema()

In [0]:
%python

ufs_validas = [
    "AC", "AL", "AP", "AM", "BA", "CE",
    "DF", "ES", "GO", "MA", "MT", "MS",
    "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC",
    "SP", "SE", "TO"
]


df_lojas_silver = (
    df_lojas_bronze

        # ====================================================
        # REGRA TÉCNICA 1
        # id_loja não pode ser nulo nem duplicado
        # ====================================================

        .filter(
            col("id_loja").isNotNull()
        )

        .dropDuplicates(
            ["id_loja"]
        )


        # ====================================================
        # REGRA TÉCNICA 2
        # CNPJ deve possuir 14 dígitos numéricos e ser único
        # ====================================================

        .withColumn(
            "cnpj",
            regexp_replace(
                col("cnpj").cast("string"),
                "[^0-9]",
                ""
            )
        )

        .filter(
            length(col("cnpj")) == 14
        )

        .dropDuplicates(
            ["cnpj"]
        )


        # ====================================================
        # REGRA TÉCNICA 3
        # UF válida
        # ====================================================

        .withColumn(
            "estado_loja",
            upper(
                trim(
                    col("estado_loja")
                )
            )
        )

        .filter(
            col("estado_loja")
                .isin(ufs_validas)
        )


        # ====================================================
        # AUDITORIA SILVER
        # ====================================================

        .withColumn(
            "silver_processed_at",
            current_timestamp()
        )

        .withColumn(
            "ano",
            year("silver_processed_at")
        )

        .withColumn(
            "mes",
            month("silver_processed_at")
        )
)


print(
    "Silver physical_lojas:",
    df_lojas_silver.count()
)

display(df_lojas_silver)

In [0]:
%python

print(
    "id_loja nulo:",
    df_lojas_silver
        .filter(col("id_loja").isNull())
        .count()
)

print(
    "id_loja distintos:",
    df_lojas_silver
        .select("id_loja")
        .distinct()
        .count()
)

print(
    "CNPJ inválido:",
    df_lojas_silver
        .filter(length(col("cnpj")) != 14)
        .count()
)

print(
    "CNPJ distintos:",
    df_lojas_silver
        .select("cnpj")
        .distinct()
        .count()
)

print(
    "UF inválida:",
    df_lojas_silver
        .filter(
            ~col("estado_loja").isin(ufs_validas)
        )
        .count()
)

In [0]:
%python

df_produtos_bronze = (
    spark.read
        .format("delta")
        .options(**config)
        .load(bronze_produtos)
)

print(
    "Bronze physical_produtos_pereciveis:",
    df_produtos_bronze.count()
)

df_produtos_bronze.printSchema()

In [0]:
%python

colunas_obrigatorias = [
    "sku",
    "unidade_medida",
    "preco_lista",
    "categoria_pai"
]

colunas_ausentes = [
    coluna
    for coluna in colunas_obrigatorias
    if coluna not in df_produtos_bronze.columns
]

if colunas_ausentes:

    raise ValueError(
        "A Bronze não possui as colunas "
        "obrigatórias para aplicar todas as regras: "
        + ", ".join(colunas_ausentes)
    )

print(
    "Schema validado. "
    "Todas as colunas obrigatórias existem."
)


In [0]:
%python

categorias_pereciveis = [
    "Hortifruti",
    "Laticínios & Ovos",
    "Frios & Embutidos",
    "Açougue & Peixaria",
    "Congelados",
    "Padaria Fresca"
]


df_produtos_silver = (
    df_produtos_bronze


        # ====================================================
        # REGRA TÉCNICA 1
        # sku não pode ser nulo nem duplicado
        # ====================================================

        .filter(
            col("sku").isNotNull()
        )

        .dropDuplicates(
            ["sku"]
        )


        # ====================================================
        # REGRA TÉCNICA 2
        # unidade_medida ∈ kg, L, un
        # ====================================================

        .filter(
            col("unidade_medida").isin(
                "kg",
                "L",
                "un"
            )
        )


        # ====================================================
        # REGRA TÉCNICA 3
        # 0 < preco_lista <= 299.99
        # ====================================================

        .filter(
            (col("preco_lista") > 0)
            &
            (col("preco_lista") <= 299.99)
        )


        # ====================================================
        # REGRA TÉCNICA 4
        # categoria_pai válida
        # ====================================================

        .filter(
            col("categoria_pai")
                .isin(categorias_pereciveis)
        )

        # ====================================================
        # AUDITORIA SILVER
        # ====================================================

        .withColumn(
            "silver_processed_at",
            current_timestamp()
        )

        .withColumn(
            "ano",
            year("silver_processed_at")
        )

        .withColumn(
            "mes",
            month("silver_processed_at")
        )
)


print(
    "Silver physical_produtos_pereciveis:",
    df_produtos_silver.count()
)

display(df_produtos_silver)

In [0]:
%python

print(
    "SKU nulo:",
    df_produtos_silver
        .filter(col("sku").isNull())
        .count()
)

print(
    "SKUs:",
    df_produtos_silver.count()
)

print(
    "SKUs distintos:",
    df_produtos_silver
        .select("sku")
        .distinct()
        .count()
)

print(
    "Unidade inválida:",
    df_produtos_silver
        .filter(
            ~col("unidade_medida")
                .isin("kg", "L", "un")
        )
        .count()
)

print(
    "Preço inválido:",
    df_produtos_silver
        .filter(
            (col("preco_lista") <= 0)
            |
            (col("preco_lista") > 299.99)
        )
        .count()
)

print(
    "Categoria inválida:",
    df_produtos_silver
        .filter(
            ~col("categoria_pai")
                .isin(categorias_pereciveis)
        )
        .count()
)

In [0]:
%python

caminho_silver = (
    "abfss://squad3@internshipdatalake.dfs.core.windows.net/"
    "silver/"
)

df_arquivos_silver = (
    spark.read
        .format("binaryFile")
        .options(**config)
        .option("recursiveFileLookup", "true")
        .load(caminho_silver)
        .select(
            "path",
            "modificationTime"
        )
)

display(
    df_arquivos_silver.filter(
        col("path").contains("physical_vendas_caixa")
        |
        col("path").contains("physical_itens_venda_caixa")
        |
        col("path").contains("food_estoque_lojas")
        |
        col("path").contains("food_lotes_producao")
    )
)

## Dependências para aplicação das regras de negócio
### Situação atual

| Dependência | Origem utilizada |
|---|---|
| food_lotes_producao | Silver |
| physical_vendas_caixa | Raw - temporário |
| physical_itens_venda_caixa | Raw - temporário |
| food_estoque_lojas | Raw - temporário |


In [0]:
%python

# ============================================================
# DEPENDÊNCIA DISPONÍVEL NA SILVER
# ============================================================

silver_food_lotes = (
    "abfss://squad3@internshipdatalake.dfs.core.windows.net/"
    "silver/food_lotes_producao"
)


# ============================================================
# DEPENDÊNCIAS AINDA NÃO DISPONÍVEIS NA SILVER
# Consumo temporário diretamente da Raw
# ============================================================

raw_vendas_caixa = (
    "abfss://raw@internshipdatalake.dfs.core.windows.net/"
    "batch-data/physical_vendas_caixa.csv"
)

raw_itens_venda = (
    "abfss://raw@internshipdatalake.dfs.core.windows.net/"
    "batch-data/physical_itens_venda_caixa.csv"
)

raw_estoque_lojas = (
    "abfss://raw@internshipdatalake.dfs.core.windows.net/"
    "batch-data/food_estoque_lojas.csv"
)

print("Caminhos das dependências definidos.")

In [0]:
%python

df_lotes = (
    spark.read
        .format("delta")
        .options(**config)
        .load(silver_food_lotes)
)

print("==========================================")
print("FONTE: SILVER")
print("Tabela: food_lotes_producao")
print("==========================================")

print(
    "Total de registros:",
    df_lotes.count()
)

df_lotes.printSchema()

display(
    df_lotes.limit(10)
)

In [0]:
%python

df_vendas = (
    spark.read
        .format("csv")
        .options(**config)
        .option("header", "true")
        .option("inferSchema", "true")
        .load(raw_vendas_caixa)
)

print("==========================================")
print("FONTE: RAW - TEMPORÁRIA")
print("Tabela: physical_vendas_caixa")
print("==========================================")

print(
    "Total de registros:",
    df_vendas.count()
)

df_vendas.printSchema()

display(
    df_vendas.limit(10)
)

In [0]:
%python
raw_itens_venda = (
    "abfss://raw@internshipdatalake.dfs.core.windows.net/"
    "batch-data/physical_itens_venda_caixa.csv"
)

In [0]:
%python

df_itens_venda = (
    spark.read
        .format("csv")
        .options(**config)
        .option("header", "true")
        .option("inferSchema", "true")
        .load(raw_itens_venda)
)

print("==========================================")
print("FONTE: RAW - TEMPORÁRIA")
print("Tabela: physical_itens_venda_caixa")
print("==========================================")

print(
    "Total de registros:",
    df_itens_venda.count()
)

df_itens_venda.printSchema()

display(
    df_itens_venda.limit(10)
)

In [0]:
%python

df_estoque = (
    spark.read
        .format("csv")
        .options(**config)
        .option("header", "true")
        .option("inferSchema", "true")
        .load(raw_estoque_lojas)
)

print("==========================================")
print("FONTE: RAW - TEMPORÁRIA")
print("Tabela: food_estoque_lojas")
print("==========================================")

print(
    "Total de registros:",
    df_estoque.count()
)

df_estoque.printSchema()

display(
    df_estoque.limit(10)
)

In [0]:
%python

print("==============================================")
print("DEPENDÊNCIAS CARREGADAS")
print("==============================================")

print(
    "food_lotes_producao:",
    df_lotes.count(),
    "registros | Fonte: SILVER"
)

print(
    "physical_vendas_caixa:",
    df_vendas.count(),
    "registros | Fonte: RAW temporária"
)

print(
    "physical_itens_venda_caixa:",
    df_itens_venda.count(),
    "registros | Fonte: RAW temporária"
)

print(
    "food_estoque_lojas:",
    df_estoque.count(),
    "registros | Fonte: RAW temporária"
)

In [0]:
%python

print("\n==========================================")
print("food_lotes_producao - SILVER")
print("==========================================")
df_lotes.printSchema()


print("\n==========================================")
print("physical_vendas_caixa - RAW")
print("==========================================")
df_vendas.printSchema()


print("\n==========================================")
print("physical_itens_venda_caixa - RAW")
print("==========================================")
df_itens_venda.printSchema()


print("\n==========================================")
print("food_estoque_lojas - RAW")
print("==========================================")
df_estoque.printSchema()

In [0]:
%python

from pyspark.sql.functions import (
    col,
    year,
    month,
    sum as spark_sum,
    countDistinct,
    lag,
    when,
    lit,
    avg,
    datediff
)

from pyspark.sql.window import Window


receita_loja_ano = (
    df_vendas
        .filter(
            col("id_loja").isNotNull()
            & col("dt_venda").isNotNull()
            & col("valor_total_venda").isNotNull()
        )

        .withColumn(
            "ano_venda",
            year("dt_venda")
        )

        .groupBy(
            "id_loja",
            "ano_venda"
        )

        .agg(
            spark_sum("valor_total_venda")
                .alias("receita_total")
        )
)

display(
    receita_loja_ano
        .orderBy(
            "ano_venda",
            col("receita_total").desc()
        )
)

In [0]:
%python

window_yoy = (
    Window
        .partitionBy("id_loja")
        .orderBy("ano_venda")
)


crescimento_yoy_lojas = (
    receita_loja_ano

        .withColumn(
            "receita_ano_anterior",
            lag("receita_total")
                .over(window_yoy)
        )

        .withColumn(
            "crescimento_yoy_pct",

            when(
                col("receita_ano_anterior").isNull()
                |
                (col("receita_ano_anterior") == 0),

                None
            )

            .otherwise(
                (
                    (
                        col("receita_total")
                        -
                        col("receita_ano_anterior")
                    )
                    /
                    col("receita_ano_anterior")
                ) * 100
            )
        )

        .join(
            df_lojas_silver.select(
                "id_loja",
                "nome_loja"
            ),
            on="id_loja",
            how="left"
        )

        .orderBy(
            "id_loja",
            "ano_venda"
        )
)

display(crescimento_yoy_lojas)

In [0]:
%python

transacoes_loja_mes = (
    df_vendas

        .filter(
            col("id_loja").isNotNull()
            & col("id_transacao").isNotNull()
            & col("dt_venda").isNotNull()
        )

        .withColumn(
            "ano_venda",
            year("dt_venda")
        )

        .withColumn(
            "mes_venda",
            month("dt_venda")
        )

        .groupBy(
            "id_loja",
            "ano_venda",
            "mes_venda"
        )

        .agg(
            countDistinct("id_transacao")
                .alias("qtd_transacoes")
        )

        .join(
            df_lojas_silver.select(
                "id_loja",
                "nome_loja"
            ),
            on="id_loja",
            how="left"
        )

        .orderBy(
            "ano_venda",
            "mes_venda",
            "id_loja"
        )
)

display(transacoes_loja_mes)

In [0]:
%python

ticket_medio_loja_semestre = (
    df_vendas

        .filter(
            col("id_loja").isNotNull()
            & col("id_transacao").isNotNull()
            & col("dt_venda").isNotNull()
            & col("valor_total_venda").isNotNull()
        )

        .withColumn(
            "ano_venda",
            year("dt_venda")
        )

        .withColumn(
            "semestre",
            when(
                month("dt_venda") <= 6,
                1
            ).otherwise(2)
        )

        .groupBy(
            "id_loja",
            "ano_venda",
            "semestre"
        )

        .agg(
            spark_sum("valor_total_venda")
                .alias("receita_semestre"),

            countDistinct("id_transacao")
                .alias("qtd_transacoes")
        )

        .withColumn(
            "ticket_medio",
            col("receita_semestre")
            /
            col("qtd_transacoes")
        )

        .join(
            df_lojas_silver.select(
                "id_loja",
                "nome_loja"
            ),
            on="id_loja",
            how="left"
        )

        .orderBy(
            "ano_venda",
            "semestre",
            "id_loja"
        )
)

display(ticket_medio_loja_semestre)

In [0]:
%python

skus_ativos_categoria_mes = (
    df_estoque.alias("e")

        .join(
            df_produtos_silver
                .select(
                    "sku",
                    "categoria_pai"
                )
                .alias("p"),

            col("e.sku") == col("p.sku"),

            "inner"
        )

        .filter(
            col("e.dt_snapshot").isNotNull()
        )

        .withColumn(
            "ano_snapshot",
            year("e.dt_snapshot")
        )

        .withColumn(
            "mes_snapshot",
            month("e.dt_snapshot")
        )

        .groupBy(
            "ano_snapshot",
            "mes_snapshot",
            "categoria_pai"
        )

        .agg(
            countDistinct("e.sku")
                .alias("qtd_skus_ativos")
        )

        .orderBy(
            "ano_snapshot",
            "mes_snapshot",
            "categoria_pai"
        )
)

display(skus_ativos_categoria_mes)

In [0]:
%python

validade_media_categoria = (
    df_lotes.alias("l")

        .join(
            df_produtos_silver
                .select(
                    "sku",
                    "categoria_pai"
                )
                .alias("p"),

            col("l.sku") == col("p.sku"),

            "inner"
        )

        .filter(
            col("l.dt_fabricacao").isNotNull()
            &
            col("l.dt_validade").isNotNull()
        )

        .withColumn(
            "dias_validade",
            datediff(
                col("l.dt_validade"),
                col("l.dt_fabricacao")
            )
        )

        .filter(
            col("dias_validade") >= 0
        )

        .groupBy(
            "categoria_pai"
        )

        .agg(
            avg("dias_validade")
                .alias("validade_media_dias")
        )

        .orderBy(
            "categoria_pai"
        )
)

display(validade_media_categoria)

## Validação de SKUs

`food_estoque_lojas` possui `sku` e pode ser validada diretamente.

`physical_itens_venda_caixa` possui apenas `codigo_barras_produto`.  
Por isso, precisamos localizar uma tabela que permita o relacionamento:

`codigo_barras_produto` → `sku`

A busca abaixo identifica essa dependência no Data Lake.

## Exploração das tabelas envolvidas na validação de SKUs

Antes de definir o relacionamento entre as tabelas, serão analisados o schema,
os registros e os valores das fontes envolvidas para identificar possíveis
chaves de correspondência.

In [0]:
%python

print("==========================================")
print("PHYSICAL_PRODUTOS_PERECIVEIS")
print("==========================================")

print("Registros:", df_produtos_silver.count())
print("Colunas:", df_produtos_silver.columns)

df_produtos_silver.printSchema()

display(
    df_produtos_silver.limit(30)
)


print("\n==========================================")
print("PHYSICAL_ITENS_VENDA_CAIXA")
print("==========================================")

print("Registros:", df_itens_venda.count())
print("Colunas:", df_itens_venda.columns)

df_itens_venda.printSchema()

display(
    df_itens_venda.limit(30)
)


print("\n==========================================")
print("FOOD_ESTOQUE_LOJAS")
print("==========================================")

print("Registros:", df_estoque.count())
print("Colunas:", df_estoque.columns)

df_estoque.printSchema()

display(
    df_estoque.limit(30)
)

## Validação das chaves de produto

Será verificada a correspondência entre os identificadores de produto
disponíveis nas tabelas envolvidas:

- `physical_produtos_pereciveis.sku`
- `food_estoque_lojas.sku`
- `physical_itens_venda_caixa.codigo_barras_produto`

O objetivo é identificar se os campos podem ser relacionados diretamente
antes de buscar uma tabela auxiliar de mapeamento.

In [0]:
%python

# ============================================================
# IDENTIFICADORES DO CATÁLOGO
# ============================================================

catalogo = (
    df_produtos_silver
        .select(
            F.trim(
                F.col("sku").cast("string")
            ).alias("identificador")
        )
        .filter(F.col("identificador").isNotNull())
        .distinct()
)


# ============================================================
# IDENTIFICADORES DO ESTOQUE
# ============================================================

estoque = (
    df_estoque
        .select(
            F.trim(
                F.col("sku").cast("string")
            ).alias("identificador")
        )
        .filter(F.col("identificador").isNotNull())
        .distinct()
)


# ============================================================
# IDENTIFICADORES DOS ITENS DE VENDA
# ============================================================

itens = (
    df_itens_venda
        .select(
            F.trim(
                F.col("codigo_barras_produto").cast("string")
            ).alias("identificador")
        )
        .filter(F.col("identificador").isNotNull())
        .distinct()
)


# ============================================================
# SKU ESTOQUE x SKU CATÁLOGO
# ============================================================

estoque_encontrados = (
    estoque
        .join(
            catalogo,
            "identificador",
            "inner"
        )
)

estoque_nao_encontrados = (
    estoque
        .join(
            catalogo,
            "identificador",
            "left_anti"
        )
)


# ============================================================
# CÓDIGO DE BARRAS x SKU CATÁLOGO
# ============================================================

itens_encontrados = (
    itens
        .join(
            catalogo,
            "identificador",
            "inner"
        )
)

itens_nao_encontrados = (
    itens
        .join(
            catalogo,
            "identificador",
            "left_anti"
        )
)


# ============================================================
# RESULTADO
# ============================================================

print("===== ESTOQUE x CATÁLOGO =====")

print(
    "SKUs distintos no estoque:",
    estoque.count()
)

print(
    "Encontrados no catálogo:",
    estoque_encontrados.count()
)

print(
    "Não encontrados:",
    estoque_nao_encontrados.count()
)


print("\n===== ITENS DE VENDA x CATÁLOGO =====")

print(
    "Códigos de barras distintos:",
    itens.count()
)

print(
    "Correspondem a SKU do catálogo:",
    itens_encontrados.count()
)

print(
    "Não correspondem:",
    itens_nao_encontrados.count()
)

## Validação com catálogo seco

O catálogo `physical_produtos_pereciveis` possui 765 SKUs que foram
identificados nas tabelas de estoque e itens de venda.

Os identificadores restantes serão comparados com `ecommerce_produtos`,
que representa o catálogo seco, para verificar se pertencem a outro
catálogo e não a uma inconsistência dos produtos perecíveis.

In [0]:
%python

display(
    df_arquivos_silver.filter(
        F.lower(F.col("path")).contains("ecommerce_produtos")
    )
    .select(
        "path",
        "modificationTime"
    )
    .distinct()
)

In [0]:
%python

raw_itens_venda = (
    "abfss://raw@internshipdatalake.dfs.core.windows.net/"
    "batch-data/physical_itens_venda_caixa.csv"
)

df_itens_venda = (
    spark.read
        .format("csv")
        .options(**config)
        .option("header", "true")
        .option("inferSchema", "true")
        .load(raw_itens_venda)
)

print("Total de registros:", df_itens_venda.count())

df_itens_venda.printSchema()

display(df_itens_venda.limit(10))

In [0]:
%python

caminho_ecommerce_raw = (
    "abfss://raw@internshipdatalake.dfs.core.windows.net/"
    "batch-data/ecommerce_produtos.csv"
)

df_ecommerce = (
    spark.read
        .format("csv")
        .options(**config)
        .option("header", "true")
        .option("inferSchema", "true")
        .load(caminho_ecommerce_raw)
)

print("Fonte: RAW - temporária")
print("Registros:", df_ecommerce.count())

df_ecommerce.printSchema()

display(df_ecommerce.limit(20))

In [0]:
%python

# ============================================================
# IDENTIFICADORES DISTINTOS
# ============================================================

sku_estoque = (
    df_estoque
        .select(
            F.trim(
                F.col("sku").cast("string")
            ).alias("identificador")
        )
        .filter(F.col("identificador").isNotNull())
        .distinct()
)

codigo_itens = (
    df_itens_venda
        .select(
            F.trim(
                F.col("codigo_barras_produto").cast("string")
            ).alias("identificador")
        )
        .filter(F.col("identificador").isNotNull())
        .distinct()
)


# ============================================================
# PRESENTES NAS DUAS TABELAS
# ============================================================

presentes_nas_duas = (
    sku_estoque
        .join(
            codigo_itens,
            on="identificador",
            how="inner"
        )
)


# ============================================================
# SOMENTE NO ESTOQUE
# ============================================================

somente_estoque = (
    sku_estoque
        .join(
            codigo_itens,
            on="identificador",
            how="left_anti"
        )
)


# ============================================================
# SOMENTE NOS ITENS DE VENDA
# ============================================================

somente_itens = (
    codigo_itens
        .join(
            sku_estoque,
            on="identificador",
            how="left_anti"
        )
)


# ============================================================
# RESULTADO
# ============================================================

print("===== ESTOQUE x ITENS DE VENDA =====")

print(
    "SKUs distintos no estoque:",
    sku_estoque.count()
)

print(
    "Códigos distintos nos itens de venda:",
    codigo_itens.count()
)

print(
    "Identificadores presentes nas duas:",
    presentes_nas_duas.count()
)

print(
    "Somente no estoque:",
    somente_estoque.count()
)

print(
    "Somente nos itens de venda:",
    somente_itens.count()
)

In [0]:
%python

print("===== SOMENTE NO ESTOQUE =====")
display(
    somente_estoque.orderBy("identificador")
)

print("===== SOMENTE NOS ITENS DE VENDA =====")
display(
    somente_itens.orderBy("identificador")
)

In [0]:
%python

print("===== ECOMMERCE_PRODUTOS =====")
print("Registros:", df_ecommerce.count())
print("Colunas:", df_ecommerce.columns)

df_ecommerce.printSchema()

display(df_ecommerce.limit(20))

In [0]:
%python

from pyspark.sql import functions as F

itens = (
    df_itens_venda
    .select(F.trim("codigo_barras_produto").alias("sku"))
    .distinct()
)

secos = (
    df_ecommerce
    .select(F.trim("sku").alias("sku"))
    .distinct()
)

itens_no_seco = itens.join(
    secos,
    "sku",
    "inner"
)

print("Itens distintos:", itens.count())
print("Encontrados no catálogo seco:", itens_no_seco.count())

### Catálogos de Produtos

| Tabela | Tipo de catálogo | Descrição |
|---|---|---|
| ecommerce_produtos | Catálogo seco | Catálogo destinado aos produtos secos. |
| physical_produtos_pereciveis | Catálogo perecível | Catálogo destinado aos produtos perecíveis das lojas físicas. |

In [0]:
%python
from pyspark.sql import functions as F

kpi_preco_medio_semestre = (
    df_produtos_silver
    .withColumn(
        "semestre",
        F.when(F.col("mes") <= 6, F.lit("1º Semestre"))
         .otherwise(F.lit("2º Semestre"))
    )
    .groupBy(
        "ano",
        "semestre",
        "subcategoria"
    )
    .agg(
        F.round(F.avg("preco_lista"), 2).alias("preco_medio")
    )
    .orderBy(
        "ano",
        "semestre",
        "subcategoria"
    )
)

display(kpi_preco_medio_semestre)

### physical_lojas

**Granularidade:** 1 linha por loja por mês.

Cada registro da tabela Gold `physical_lojas` representa uma loja em um determinado mês.

### Regra 10 — Validação das lojas na camada Gold

**Status: Pendente — etapa Gold**

Validar que todas as lojas geradas dinamicamente estejam presentes na tabela Gold de lojas.

No momento desta implementação, a tabela/camada Gold de lojas ainda não está disponível no caminho do Squad 3.  
A validação deverá ser realizada após a criação/disponibilização da tabela Gold, comparando os `id_loja` gerados com os `id_loja` presentes na Gold.

**Critério de sucesso:** nenhuma loja gerada pode estar ausente da tabela Gold.

In [0]:
%python

display(
    df_lojas_silver
    .select("cidade_loja")
    .distinct()
    .orderBy("cidade_loja")
)

In [0]:
%python

df_lojas_silver.printSchema()

In [0]:
%python

display(
    df_lojas_silver
    .select("cidade_loja", "estado_loja")
    .distinct()
    .orderBy("estado_loja", "cidade_loja")
)

In [0]:
%python
import requests

url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

municipios_ibge = requests.get(url).json()

print("Municípios retornados pelo IBGE:", len(municipios_ibge))
print(municipios_ibge[0])

In [0]:
%python
from pyspark.sql import functions as F

lista_municipios = [
    (
        m["id"],
        m["nome"],
        m["regiao-imediata"]["regiao-intermediaria"]["UF"]["sigla"]
    )
    for m in municipios_ibge
    if m.get("regiao-imediata") is not None
]

df_municipios_ibge = spark.createDataFrame(
    lista_municipios,
    ["codigo_ibge", "municipio_ibge", "uf_ibge"]
)

cidades_lojas = (
    df_lojas_silver
    .select("cidade_loja", "estado_loja")
    .distinct()
)

validacao_municipios = (
    cidades_lojas.alias("loja")
    .join(
        df_municipios_ibge.alias("ibge"),
        (F.lower(F.trim(F.col("loja.cidade_loja"))) ==
         F.lower(F.trim(F.col("ibge.municipio_ibge"))))
        &
        (F.col("loja.estado_loja") == F.col("ibge.uf_ibge")),
        "left"
    )
    .select(
        "cidade_loja",
        "estado_loja",
        "codigo_ibge",
        "municipio_ibge"
    )
    .orderBy("estado_loja", "cidade_loja")
)

display(validacao_municipios)

In [0]:
%python
from pyspark.sql import functions as F

validacao_municipios = (
    validacao_municipios
    .withColumn(
        "codigo_ibge",
        F.when(F.col("cidade_loja") == "Barra de São João", 3301306)
         .when(F.col("cidade_loja") == "Cunhambebe", 3300100)
         .when(F.col("cidade_loja") == "Porto das Caixas", 3301900)
         .when(F.col("cidade_loja") == "Riograndina", 3303401)
         .otherwise(F.col("codigo_ibge"))
    )
    .withColumn(
        "municipio_ibge",
        F.when(F.col("cidade_loja") == "Barra de São João", "Casimiro de Abreu")
         .when(F.col("cidade_loja") == "Cunhambebe", "Angra dos Reis")
         .when(F.col("cidade_loja") == "Porto das Caixas", "Itaboraí")
         .when(F.col("cidade_loja") == "Riograndina", "Nova Friburgo")
         .otherwise(F.col("municipio_ibge"))
    )
)

display(
    validacao_municipios.orderBy("estado_loja", "cidade_loja")
)

In [0]:
%python
import requests

codigos_ibge = [
    str(row["codigo_ibge"])
    for row in validacao_municipios
        .select("codigo_ibge")
        .distinct()
        .collect()
]

dados_populacao = []

for codigo in codigos_ibge:
    url = (
        f"https://servicodados.ibge.gov.br/api/v3/"
        f"agregados/6579/periodos/-1/variaveis/9324"
        f"?localidades=N6[{codigo}]"
    )

    resposta = requests.get(url)
    
    if resposta.status_code == 200:
        resultado = resposta.json()

        if resultado:
            dados_populacao.append(
                (codigo, resultado)
            )

print("Municípios consultados:", len(codigos_ibge))
print("Respostas obtidas:", len(dados_populacao))

In [0]:
%python

populacao_lista = []

for codigo, resultado in dados_populacao:
    serie = resultado[0]["resultados"][0]["series"][0]["serie"]

    periodo = list(serie.keys())[0]
    populacao = serie[periodo]

    populacao_lista.append(
        (int(codigo), int(populacao))
    )

df_populacao_ibge = spark.createDataFrame(
    populacao_lista,
    ["codigo_ibge", "populacao"]
)

df_populacao_ibge = (
    validacao_municipios
    .select(
        "codigo_ibge",
        "municipio_ibge",
        "estado_loja"
    )
    .join(
        df_populacao_ibge,
        "codigo_ibge",
        "left"
    )
    .select(
        "municipio_ibge",
        "estado_loja",
        "codigo_ibge",
        "populacao"
    )
    .orderBy("estado_loja", "municipio_ibge")
)

display(df_populacao_ibge)

In [0]:
%python
import requests

codigo_teste = 3200409

url_renda = (
    f"https://apisidra.ibge.gov.br/values/"
    f"t/10295/n6/{codigo_teste}/v/13431/p/2022"
)

resposta_renda = requests.get(url_renda)

print("Status:", resposta_renda.status_code)
print(resposta_renda.text[:2000])

In [0]:
%python
import requests

renda_lista = []

for row in (
    validacao_municipios
    .select("codigo_ibge", "municipio_ibge", "estado_loja")
    .distinct()
    .collect()
):
    
    codigo = row["codigo_ibge"]
    municipio = row["municipio_ibge"]
    uf = row["estado_loja"]

    url = (
        f"https://apisidra.ibge.gov.br/values/"
        f"t/10295/n6/{codigo}/v/13431/p/2022"
    )

    resposta = requests.get(url)

    if resposta.status_code == 200:
        dados = resposta.json()

        # primeira posição é o cabeçalho
        if len(dados) > 1:
            renda = dados[1]["V"]

            renda_lista.append(
                (
                    int(codigo),
                    municipio,
                    uf,
                    float(renda)
                )
            )

df_renda_ibge = spark.createDataFrame(
    renda_lista,
    [
        "codigo_ibge",
        "municipio_ibge",
        "estado_loja",
        "renda_per_capita"
    ]
)

display(
    df_renda_ibge
    .orderBy("estado_loja", "municipio_ibge")
)

In [0]:
%python
df_demografia_ibge = (
    df_populacao_ibge
    .join(
        df_renda_ibge.select(
            "codigo_ibge",
            "renda_per_capita"
        ),
        on="codigo_ibge",
        how="left"
    )
    .select(
        "codigo_ibge",
        "municipio_ibge",
        "estado_loja",
        "populacao",
        "renda_per_capita"
    )
    .orderBy("estado_loja", "municipio_ibge")
)

display(df_demografia_ibge)

print("Total de municípios:", df_demografia_ibge.count())
print(
    "Sem renda:",
    df_demografia_ibge.filter(F.col("renda_per_capita").isNull()).count()
)
print(
    "Sem população:",
    df_demografia_ibge.filter(F.col("populacao").isNull()).count()
)